In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 44.2 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import optuna
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split,KFold
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Device: cuda


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CSV_PATH = "/content/drive/MyDrive/ASQP/data/augmented/treino_overs.csv"

df = pd.read_csv(CSV_PATH)

TEXT_COL = "text"
TARGET_COL = "target"

if "input" not in df.columns:
    df["input"] = "extrair quadruplas: " + df[TEXT_COL].astype(str)

df = df.dropna(subset=["input", TARGET_COL]).copy()

df["input"] = df["input"].astype(str)
df[TARGET_COL] = df[TARGET_COL].astype(str)

print("Total:", len(df))
display(df[["input", TARGET_COL]].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total: 2190


,input,target
0,extrair quadruplas:apesar das qualificações na...,[A] atendentes [C] service [O] atenciosos [P] ...
1,extrair quadruplas:hotel super chamoso e bem l...,[A] localizado [C] location [O] bem [P] pos [S...
2,extrair quadruplas:o hotel de sevres é um hote...,[A] zona [C] location [O] sossegada [P] pos [S...
3,extrair quadruplas:viagem em familia em que se...,[A] localização [C] location [O] excelente [P]...
4,extrair quadruplas:no meio da strip ao lado do...,[A] quarto [C] structure [O] amplo [P] pos [SS...


# Hyperparams Search - Raw x MT5

In [ ]:
def tokenize_batch(batch):
  model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
  )

  labels = tokenizer(
        batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
  )

  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [ ]:
MODEL_NAME = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SPECIAL_TOKENS = ["[A]", "[C]", "[O]", "[P]", "[SSEP]"]
tokenizer.add_tokens(SPECIAL_TOKENS)

print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer carregado.
Tamanho do vocabulário: 250105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 358
MAX_TARGET_LEN: 152


In [ ]:
def clean_pred(text):
  if not isinstance(text, str):
    return ""

  text = text.replace("<extra_id_0>", "")
  text = text.replace("<extra_id_1>", "")
  text = text.replace("<pad>", "")
  text = text.replace("</s>", "")
  text = " ".join(text.split())

  return text.strip()


def parse_target_string(text):
  if not isinstance(text, str):
    return set()

  text = clean_pred(text)

  quads = []

  for part in text.split("[SSEP]"):
      part = part.strip()

      if not part:
        continue

      try:
        a = part.split("[A]")[1].split("[C]")[0].strip()
        c = part.split("[C]")[1].split("[O]")[0].strip()
        o = part.split("[O]")[1].split("[P]")[0].strip()
        p = part.split("[P]")[1].strip()

        quads.append((a, c, o, p))

      except Exception:
        continue

  return set(quads)


def evaluate_asqp(golds, preds):
  total_correct = 0
  total_pred = 0
  total_gold = 0

  for gold, pred in zip(golds, preds):
    gold_set = parse_target_string(gold)
    pred_set = parse_target_string(pred)

    total_correct += len(gold_set & pred_set)
    total_pred += len(pred_set)
    total_gold += len(gold_set)

  precision = total_correct / total_pred if total_pred else 0
  recall = total_correct / total_gold if total_gold else 0

  f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
  )

  return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "correct": total_correct,
        "pred_total": total_pred,
        "gold_total": total_gold
    }

In [ ]:
def generate_predictions(model, tokenizer, texts):
  model.eval()
  preds = []

  for text in texts:
    inputs = tokenizer(
          text,
          return_tensors="pt",
          max_length=MAX_INPUT_LEN,
          truncation=True
    ).to(model.device)

    with torch.no_grad():
      outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True)

      pred = tokenizer.decode(outputs[0], skip_special_tokens=False)
      pred = clean_pred(pred)

      preds.append(pred)

  return preds

In [ ]:
def objective(trial):
  clear_memory()

  learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        5e-4,
        log=True
  )

  num_train_epochs = 6

  weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 0.001, 0.01]
  )

  warmup_ratio = trial.suggest_categorical(
        "warmup_ratio",
        [0.0, 0.05, 0.1]
  )

  kf = KFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED
  )

  fold_f1s = []

  for fold, (train_idx, val_idx) in enumerate(kf.split(df), start=1):
      print(f"\nTrial {trial.number} | Fold {fold}/3")

      train_fold = df.iloc[train_idx].reset_index(drop=True)
      val_fold = df.iloc[val_idx].reset_index(drop=True)

      train_dataset = Dataset.from_pandas(train_fold)
      val_dataset = Dataset.from_pandas(val_fold)

      train_tokenized = train_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=train_dataset.column_names
      )

      val_tokenized = val_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=val_dataset.column_names
      )

      if MODEL_NAME == "unicamp-dl/ptt5-base-portuguese-vocab":

        model = T5ForConditionalGeneration.from_pretrained("unicamp-dl/ptt5-base-portuguese-vocab")
      else:

        model = AutoModelForSeq2SeqLM.from_pretrained(
              MODEL_NAME,
              tie_word_embeddings=False
        )

      model.resize_token_embeddings(len(tokenizer))
      model.to(DEVICE)

      data_collator = DataCollatorForSeq2Seq(
          tokenizer=tokenizer,
          model=model,
          label_pad_token_id=-100
      )

      fold_out_dir = f"/content/optuna_trial_{trial.number}_fold_{fold}"

      training_args = Seq2SeqTrainingArguments(
            output_dir=fold_out_dir,

            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,

            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            gradient_accumulation_steps= 1,

            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,

            predict_with_generate=True,
            generation_max_length=MAX_TARGET_LEN,

            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",

            fp16=False,
            report_to="none",

            seed=SEED,
            data_seed=SEED
      )

      trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=train_tokenized,
            eval_dataset=val_tokenized,
            data_collator=data_collator
      )

      trainer.train()

      preds = generate_predictions(
            model=model,
            tokenizer=tokenizer,
            texts=val_fold["input"].tolist()
      )

      golds = val_fold["target"].tolist()

      metrics = evaluate_asqp(golds, preds)
      fold_f1 = metrics["f1"]

      print("Fold metrics:", metrics)

      fold_f1s.append(fold_f1)

      del model
      del trainer
      del train_tokenized
      del val_tokenized
      clear_memory()

  mean_f1 = float(np.mean(fold_f1s))

  print(f"\nTrial {trial.number} | Mean CV F1:", mean_f1)

  return mean_f1

In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=6
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor F1 médio 3-fold:")
print(study.best_value)

[I 2026-06-25 01:39:45,530] A new study created in memory with name: no-name-53660cf5-a166-492d-a5b3-b55353147b1c



Trial 0 | Fold 1/3


Map:   0%|          | 0/1460 [00:00<?, ? examples/s]

Map:   0%|          | 0/730 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,4.144423,0.483991
2,0.559607,0.340617
3,0.392786,0.291964
4,0.309295,0.253798
5,0.263343,0.241568
6,0.223254,0.231403


Fold metrics: {'precision': 0.7116891256659355, 'recall': 0.6685310568148366, 'f1': 0.6894353369763205, 'correct': 2271, 'pred_total': 3191, 'gold_total': 3397}


[I 2026-06-25 08:51:12,414] Trial 5 finished with value: 0.6953462358515153 and parameters: {'learning_rate': 0.00017538439673163097, 'weight_decay': 0.0, 'warmup_ratio': 0.0}. Best is trial 1 with value: 0.7431789155164593.



Trial 5 | Mean CV F1: 0.6953462358515153
Melhores hiperparâmetros:
{'learning_rate': 0.0003254527469136889, 'weight_decay': 0.01, 'warmup_ratio': 0.05}
Melhor F1 médio 3-fold:
0.7431789155164593


In [ ]:
optuna_results = study.trials_dataframe()

OPTUNA_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_mt5_base_raw3x.csv"

optuna_results.to_csv(OPTUNA_PATH, index=False)

display(optuna_results)

print("Resultados salvos em:", OPTUNA_PATH)

,number,value,datetime_start,datetime_complete,duration,params_learning_rate,params_warmup_ratio,params_weight_decay,state
0,0,0.732856,2026-06-25 01:39:45.531792,2026-06-25 02:51:41.648046,0 days 01:11:56.116254,0.000241,0.10,0.010,COMPLETE
1,1,0.743179,2026-06-25 02:51:41.649110,2026-06-25 04:00:50.422400,0 days 01:09:08.773290,0.000325,0.05,0.010,COMPLETE
2,2,0.677201,2026-06-25 04:00:50.423505,2026-06-25 05:14:03.230782,0 days 01:13:12.807277,0.000233,0.05,0.001,COMPLETE
3,3,0.704563,2026-06-25 05:14:03.231894,2026-06-25 06:26:56.563292,0 days 01:12:53.331398,0.000232,0.10,0.010,COMPLETE
4,4,0.695535,2026-06-25 06:26:56.564527,2026-06-25 07:38:00.320004,0 days 01:11:03.755477,0.000163,0.10,0.001,COMPLETE
5,5,0.695346,2026-06-25 07:38:00.321212,2026-06-25 08:51:12.414093,0 days 01:13:12.092881,0.000175,0.00,0.000,COMPLETE


Resultados salvos em: /content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_mt5_base_raw3x.csv


# Hyperparams Search - Raw x PTT5

In [ ]:
MODEL_NAME = "unicamp-dl/ptt5-base-portuguese-vocab"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens(SPECIAL_TOKENS)


print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

spiece.model:   0%|          | 0.00/756k [00:00<?, ?B/s]

Tokenizer carregado.
Tamanho do vocabulário: 32105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 277
MAX_TARGET_LEN: 171


In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=6
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor F1 médio 3-fold:")
print(study.best_value)

[I 2026-06-25 08:51:16,509] A new study created in memory with name: no-name-47a24c0a-6d23-418a-ae21-b101f276f744



Trial 0 | Fold 1/3


Map:   0%|          | 0/1460 [00:00<?, ? examples/s]

Map:   0%|          | 0/730 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,2.916418,1.432163
2,1.250840,0.811019
3,0.773486,0.529230
4,0.520518,0.439414
5,0.405859,0.375269
6,0.345904,0.358443


Fold metrics: {'precision': 0.7154102331214048, 'recall': 0.695613776861937, 'f1': 0.7053731343283582, 'correct': 2363, 'pred_total': 3303, 'gold_total': 3397}


[I 2026-06-25 15:52:00,288] Trial 5 finished with value: 0.7146112575323919 and parameters: {'learning_rate': 0.0001546931426902633, 'weight_decay': 0.001, 'warmup_ratio': 0.0}. Best is trial 0 with value: 0.8348094961146617.



Trial 5 | Mean CV F1: 0.7146112575323919
Melhores hiperparâmetros:
{'learning_rate': 0.0004943191092605203, 'weight_decay': 0.001, 'warmup_ratio': 0.0}
Melhor F1 médio 3-fold:
0.8348094961146617


In [ ]:
optuna_results = study.trials_dataframe()

OPTUNA_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_ptt5_base_raw3x.csv"

optuna_results.to_csv(OPTUNA_PATH, index=False)

display(optuna_results)

print("Resultados salvos em:", OPTUNA_PATH)

,number,value,datetime_start,datetime_complete,duration,params_learning_rate,params_warmup_ratio,params_weight_decay,state
0,0,0.834809,2026-06-25 08:51:16.510438,2026-06-25 10:01:05.551754,0 days 01:09:49.041316,0.000494,0.0,0.001,COMPLETE
1,1,0.831228,2026-06-25 10:01:05.552982,2026-06-25 11:09:55.590259,0 days 01:08:50.037277,0.000393,0.0,0.000,COMPLETE
2,2,0.828337,2026-06-25 11:09:55.591341,2026-06-25 12:19:53.916263,0 days 01:09:58.324922,0.000389,0.1,0.000,COMPLETE
3,3,0.702666,2026-06-25 12:19:53.917352,2026-06-25 13:30:47.364489,0 days 01:10:53.447137,0.000135,0.1,0.001,COMPLETE
4,4,0.701183,2026-06-25 13:30:47.365768,2026-06-25 14:41:05.329623,0 days 01:10:17.963855,0.000140,0.0,0.010,COMPLETE
5,5,0.714611,2026-06-25 14:41:05.330785,2026-06-25 15:52:00.288585,0 days 01:10:54.957800,0.000155,0.0,0.001,COMPLETE


Resultados salvos em: /content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_ptt5_base_raw3x.csv
